In [2]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [3]:
import pandas as pd
from rapidfuzz import fuzz, process
import re

## Create Dataframes

In [4]:
umd_df = pd.read_csv(APT_ROOT / "dfs/umd_cyber_events_database.csv")

# Drop unnecessary columns
drop_cols = ["nato", "eu", "shanghai_coop", "oas", "mercosur", "au", "ecowas", "asean", "opec", "gulf_coop", "g7", "g20", "aukus", "csto", "oecd", "osce", "five_eyes"]

umd_df = umd_df.drop(columns=drop_cols)

#umd_df

In [5]:
# Prep list of all known APT groups and their aliases

# Normalization helper
def normalize(s):
    return str(s).strip().lower()

# 1. Canonical names
apt_names = groups_df["name"].dropna().tolist()

# 2. All alias lists flattened
alias_lists = groups_df["aliases"].dropna().tolist()
apt_aliases = [alias for sublist in alias_lists for alias in sublist]

# 3. Combined list
all_apt_terms = apt_names + apt_aliases

# 4. Normalize
all_apt_terms_norm = [normalize(x) for x in all_apt_terms]

# 5. Deduplicate + sort
all_apt_terms_norm = sorted(set(all_apt_terms_norm))

# 6. Build alias → canonical mapping
apt_map = {}
for _, row in groups_df.iterrows():
    canon = normalize(row["name"])
    apt_map[canon] = canon
    if isinstance(row["aliases"], list):
        for alias in row["aliases"]:
            apt_map[normalize(alias)] = canon

In [ ]:
paren_regex = re.compile(r"\((.*?)\)")

# Splits the actor field into tokens
def split_actor_tokens(text):
    if not isinstance(text, str) or not text.strip():
        return [], []
    
    text_norm = normalize(text)

    # Extract text inside parentheses
    inside = paren_regex.findall(text_norm)

    # Remove parentheses content to get the "outside" content
    outside_part = paren_regex.sub(" ", text_norm)

    # Tokenize outside words
    outside_tokens = outside_part.split()

    return inside, outside_tokens

ignore_tokens = {"group", "team", "play", "unit", "actor"}

# Match tokens to the apt list
def fuzzy_match_tokens(tokens, threshold):
    matches = []

    for token in tokens:
        a = normalize(token)

        # Ignore trivial / junk tokens
        if len(token) < 3:
            continue
        if token in ignore_tokens:
            continue
        
        result = process.extractOne(
            token,
            all_apt_terms_norm,
            scorer=fuzz.token_sort_ratio
        )

        if not result:
            continue
        
        match_term, score, idx = result

        if score >= threshold:
            # Map matched alias → canonical group name
            canonical = apt_map.get(match_term)
            if canonical:
                matches.append(canonical)

    return matches if matches else None

# Match the tokens inside and outside the parentheses separately
def match_actor(actor):
    inside_tokens, outside_tokens = split_actor_tokens(actor)

    # Try parentheses first (usually highest quality)
    inside_match = fuzzy_match_tokens(inside_tokens, threshold=80)
    if inside_match:
        return inside_match[0]  # return the first canonical match
    
    # Then try tokens outside parentheses
    outside_match = fuzzy_match_tokens(outside_tokens, threshold=85)
    if outside_match:
        return outside_match[0]

    return None

umd_df["apt_group"] = umd_df["actor"].apply(match_actor)

umd_apts_df = umd_df[umd_df["apt_group"].notna()].reset_index(drop=True)

#umd_apts_df
#umd_apts_df.sample(20)

,slug,original_method,event_date,reported_date,year,month,actor,actor_type,organization,industry_code,...,org_data,cust_data,description,source_url,country,actor_country,state,county,change_log,apt_group
16,e899375a96d4c4e0,1,2015-05-31,NaN,2015,5,APT32,Nation-State,Ministry of National Assembly-Senate Relations...,92,...,NaN,NaN,Vietnamese threat actor Ocean Lotus has compro...,https://www.volexity.com/blog/2017/11/06/ocean...,Cambodia,Viet Nam,NaN,NaN,NaN,apt32
120,cfed45f0bf7ca7d2,1,2019-12-14,NaN,2019,12,FIN8,Undetermined,Unnamed gas station in North America,44,...,NaN,NaN,Payments processor VISA says North American me...,https://www.zdnet.com/article/visa-warns-of-po...,Undetermined,Undetermined,NaN,NaN,NaN,fin8
152,76106a39274425db,1,2020-07-28,NaN,2020,7,RedDelta,Nation-State,Hong Kong Study Mission to China,81,...,NaN,NaN,The Hong Kong Study Mission to China has been ...,https://www.recordedfuture.com/reddelta-target...,China,China,NaN,NaN,NaN,mustang panda
205,b44761b6459a8ce9,1,2021-12-10,NaN,2021,12,Lapsus$,Criminal,Brazil Ministry of Health,92,...,NaN,NaN,Brazil's Ministry of Health suffers a signfica...,https://www.zdnet.com/article/brazilian-minist...,Brazil,Undetermined,NaN,NaN,NaN,lapsus$
323,457b4fb80896e0fe,1,2024-02-26,NaN,2024,2,Akira,Criminal,Quik Pawn Shop,52,...,NaN,NaN,The Akira ransomware group claims responsibili...,https://thecyberexpress.com/quik-pawn-shop-cyb...,United States of America,Undetermined,Alabama,Montgomery,NaN,akira
141,6421686ff4ca4b56,1,2020-04-30,NaN,2020,4,APT23,Nation-State,Vietnamese government data center,92,...,NaN,NaN,The Chinese-linked threat actor Pirate Panda s...,https://www.anomali.com/blog/anomali-suspects-...,Viet Nam,China,NaN,NaN,NaN,putter panda
309,6b809fe26ca95675,1,2023-11-24,NaN,2023,11,Ministry of State Security's (MSS) (MUSTANG PA...,Nation-State,NXP,31,...,NaN,NaN,Threat actors from the Chimera Chinese group b...,https://www-nrc-nl.translate.goog/nieuws/2023/...,Netherlands,China,NaN,NaN,NaN,mustang panda
60,a328fc2dcfb4aa06,1,2018-07-10,NaN,2018,7,Ministry of State Security's (MSS) Hainan Stat...,Nation-State,Cambodian People's Party,81,...,NaN,NaN,A threat actor has compromised political parti...,https://www.rfa.org/english/news/cambodia/hack...,Cambodia,China,NaN,NaN,NaN,leviathan
357,206f647ad7d202ce,1,2024-09-27,NaN,2024,9,NGB 3rd Technical Surveillance Bureau (Kimsuky),Nation-State,Diehl Defence,31,...,NaN,NaN,The North Korea-linked APT Kimsuky is linked t...,https://securityaffairs.com/169162/apt/kimsuky...,Germany,Korea (the Democratic People's Republic of),NaN,NaN,NaN,kimsuky
303,78a8cc7563ad4018,1,2023-11-16,NaN,2023,11,INC,Criminal,Yamaha Motor Philippines,31,...,NaN,NaN,Yamaha Motor Co. announces that one of the ser...,https://www.bleepingcomputer.com/news/security...,Philippines,Undetermined,NaN,NaN,NaN,lazarus group


## Export Dataframes

In [21]:
#umd_apts_df.to_csv("umd_apts_df.csv",index=False)